# 08 - Frontend Recheck

Significant changes have been made in the AudioFrontend Processing which should increase processing efficiency (decimation only resampling)
In addition, configs have been changed to reduce allowable kernel size and thus convolution complexity

This notebook runs the Dataset -> Dataloader -> Frontend pipeline to time various parameters on MPS to get an updated baseline; follows logic of POC5

In [1]:
# Imports
import sys, os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import subprocess

from dnn.frontend import AudioFrontEnd
from config import FS, ASVSPOOF5_PROTOCOL_TRAIN, ASVSPOOF5_AUDIO_ROOT

%matplotlib widget

In [2]:
# params
blr_bw = 2000

# Sweep params
sweep_fs = [8000]
sweep_nch = [24, 32, 48]
sweep_Tin = [2, 3, 4]
sweep_batch = [12, 16, 24]

In [3]:
# Test

# Timing/memory sweep - AudioFrontEnd only, synthetic input (compute-realistic, no real audio needed)

def time_forward(n_channels, bandwidth, fs, T_sec, batch_size, n_runs=5, device='mps'):
    try:
        afe = AudioFrontEnd(n_channels=n_channels, bandwidth=bandwidth, fs=fs).to(device)
        x = torch.randn(batch_size, int(fs * T_sec), device=device)

        # warm-up (excluded from timing)
        _ = afe(x)
        if device == 'mps':
            torch.mps.synchronize()

        start = time.time()
        for _ in range(n_runs):
            out = afe(x)
        if device == 'mps':
            torch.mps.synchronize()
        elapsed = (time.time() - start) / n_runs

        del afe, x, out
        if device == 'mps':
            torch.mps.empty_cache()
        return elapsed
    except RuntimeError as e:
        print(f"  FAILED: nch={n_channels}, fs={fs}, T={T_sec}, batch={batch_size}: {e}")
        if device == 'mps':
            torch.mps.empty_cache()
        return None

# Caffeinate during sweep
caffeinate_proc = subprocess.Popen(["caffeinate", "-i"])

total_combos = len(sweep_fs) * len(sweep_nch) * len(sweep_Tin) * len(sweep_batch)
combo_idx = 0

results = {}
for fs in sweep_fs:
    for nch in sweep_nch:
        for T in sweep_Tin:
            for bs in sweep_batch:
                combo_idx += 1
                t = time_forward(nch, blr_bw, fs, T, bs)
                results[(fs, nch, T, bs)] = t
                status = f"{t*1000:.2f} ms" if t else "FAILED"
                pct = 100 * combo_idx / total_combos
                print(f"[{combo_idx:>4}/{total_combos} {pct:5.1f}%] fs={fs:>5} nch={nch:>3} T={T} batch={bs:>3} -> {status}")


# Kill caffeinate process
caffeinate_proc.terminate()

[   1/27   3.7%] fs= 8000 nch= 24 T=2 batch= 12 -> 45.91 ms
[   2/27   7.4%] fs= 8000 nch= 24 T=2 batch= 16 -> 61.69 ms
[   3/27  11.1%] fs= 8000 nch= 24 T=2 batch= 24 -> 90.36 ms
[   4/27  14.8%] fs= 8000 nch= 24 T=3 batch= 12 -> 73.97 ms
[   5/27  18.5%] fs= 8000 nch= 24 T=3 batch= 16 -> 96.61 ms
[   6/27  22.2%] fs= 8000 nch= 24 T=3 batch= 24 -> 141.77 ms
[   7/27  25.9%] fs= 8000 nch= 24 T=4 batch= 12 -> 89.82 ms
[   8/27  29.6%] fs= 8000 nch= 24 T=4 batch= 16 -> 117.94 ms
[   9/27  33.3%] fs= 8000 nch= 24 T=4 batch= 24 -> 212.44 ms
[  10/27  37.0%] fs= 8000 nch= 32 T=2 batch= 12 -> 56.97 ms
[  11/27  40.7%] fs= 8000 nch= 32 T=2 batch= 16 -> 75.25 ms
[  12/27  44.4%] fs= 8000 nch= 32 T=2 batch= 24 -> 110.89 ms
[  13/27  48.1%] fs= 8000 nch= 32 T=3 batch= 12 -> 91.19 ms
[  14/27  51.9%] fs= 8000 nch= 32 T=3 batch= 16 -> 121.65 ms
[  15/27  55.6%] fs= 8000 nch= 32 T=3 batch= 24 -> 188.02 ms
[  16/27  59.3%] fs= 8000 nch= 32 T=4 batch= 12 -> 111.55 ms
[  17/27  63.0%] fs= 8000 nch= 32

In [4]:
# Identify "cliff" combos - forward passes that technically completed but ran
# anomalously slow relative to a smaller-batch neighbor at the same
# (fs, n_channels, T) -- likely swapping/thrashing near the MPS memory
# ceiling, not a genuine steady-state timing.

CLIFF_THRESHOLD = 2.5  # flag if time jumps >2.5x vs. the next-smaller batch size

unusable = []

for fs in sweep_fs:
    for nch in sweep_nch:
        for T in sweep_Tin:
            # batch sizes at this (fs, nch, T), in ascending order
            batches_here = sorted(bs for (f, n, t, bs) in results if f == fs and n == nch and t == T)
            prev_time = None
            for bs in batches_here:
                t_val = results[(fs, nch, T, bs)]
                if t_val is None:
                    unusable.append({
                        "fs": fs, "n_channels": nch, "T": T, "batch": bs,
                        "time_ms": None, "reason": "OOM / exception"
                    })
                    prev_time = None  # can't compare across a failure
                    continue
                if prev_time is not None and t_val > CLIFF_THRESHOLD * prev_time:
                    unusable.append({
                        "fs": fs, "n_channels": nch, "T": T, "batch": bs,
                        "time_ms": t_val * 1000,
                        "reason": f"cliff ({t_val/prev_time:.1f}x jump vs batch={batches_here[batches_here.index(bs)-1]}"
                    })
                prev_time = t_val

print(f"{'fs':>6} {'nch':>4} {'T':>3} {'batch':>6} {'time_ms':>10}  reason")
for u in unusable:
    t_str = f"{u['time_ms']:.2f}" if u["time_ms"] is not None else "  N/A"
    print(f"{u['fs']:>6} {u['n_channels']:>4} {u['T']:>3} {u['batch']:>6} {t_str:>10}  {u['reason']}")

print(f"\nTotal unusable combos: {len(unusable)} / {len(results)}")

# Usable set = everything NOT in unusable
unusable_keys = {(u["fs"], u["n_channels"], u["T"], u["batch"]) for u in unusable}
usable_results = {k: v for k, v in results.items() if k not in unusable_keys and v is not None}
print(f"Usable combos: {len(usable_results)} / {len(results)}")

    fs  nch   T  batch    time_ms  reason

Total unusable combos: 0 / 27
Usable combos: 27 / 27


In [5]:
# Filter: usable combos = fast enough (epoch time budget) AND no anomalous cliff/failure

N_TRAIN = 182357
MAX_EPOCH_MINUTES = 15
CLIFF_THRESHOLD = 2.5

def epoch_time_minutes(t_per_step, batch_size, n_train=N_TRAIN):
    return (t_per_step * (n_train / batch_size)) / 60.0

# Identify cliff/failure keys
unusable_keys = set()
for fs in sweep_fs:
    for nch in sweep_nch:
        for T in sweep_Tin:
            batches_here = sorted(bs for (f, n, t, bs) in results if f == fs and n == nch and t == T)
            prev_time = None
            for bs in batches_here:
                t_val = results[(fs, nch, T, bs)]
                if t_val is None:
                    unusable_keys.add((fs, nch, T, bs))
                    prev_time = None
                    continue
                if prev_time is not None and t_val > CLIFF_THRESHOLD * prev_time:
                    unusable_keys.add((fs, nch, T, bs))
                prev_time = t_val

# Apply epoch-time + cliff filters together
usable = []
for (fs, nch, T, bs), t in results.items():
    if t is None or (fs, nch, T, bs) in unusable_keys:
        continue
    est_min = epoch_time_minutes(t, bs)
    if est_min <= MAX_EPOCH_MINUTES:
        usable.append({
            "fs": fs, "n_channels": nch, "T": T, "batch": bs,
            "time_ms": t * 1000, "est_epoch_min": est_min
        })

usable.sort(key=lambda d: d["est_epoch_min"])

print(f"{'fs':>6} {'nch':>4} {'T':>3} {'batch':>6} {'time_ms':>10} {'est_epoch_min':>14}")
for d in usable:
    print(f"{d['fs']:>6} {d['n_channels']:>4} {d['T']:>3} {d['batch']:>6} {d['time_ms']:>10.2f} {d['est_epoch_min']:>14.1f}")

print(f"\nUsable combos: {len(usable)} / {len(results)}")

    fs  nch   T  batch    time_ms  est_epoch_min
  8000   24   2     24      90.36           11.4
  8000   24   2     12      45.91           11.6
  8000   24   2     16      61.69           11.7
  8000   32   2     24     110.89           14.0
  8000   32   2     16      75.25           14.3
  8000   32   2     12      56.97           14.4

Usable combos: 6 / 27


## Findings
- Most sweep combos unusable — either OOM or 5-20x time "cliffs" (MPS memory thrashing)
- Best usable: nch=24, T=2, fs=8000, batch=24 → ~11.4 min/epoch (frontend only)
nch=48, T≥3 consistently too slow/unstable
- Scope response: reduce T to 2s (stricter than AASIST-L, not a shortcut), add per-epoch data subsampling
- fs/BLR bandwidth unchanged — leakage margins are absolute-Hz, no re-validation needed
- n_channels=24 likely the practical starting point going forward

- Potentially worth re-parameterizing BLR (current BW - new fs after resample - is significant portion of usable freq range)
    - 1 kHz BW divides evenly into any probably samplerate and reduces data load at output, but makes chirplet gen more rigid
    - keep 2kHz BW until after training tests